# LangChain Middleware

এই notebook-এ LangChain-এর **Middleware** system দেখানো হবে।

Middleware হলো এমন একটা layer যেটা LLM call-এর **আগে** বা **পরে** automatically execute হয় — বিনা কোনো manual code ছাড়া।

```
User Input
    ↓
[ Middleware: before_model ]   ← summarization, validation, PII redaction
    ↓
  LLM Call
    ↓
[ Middleware: after_model ]    ← logging, human approval
    ↓
Agent Output
```

| Middleware | কাজ | কখন trigger হয় |
|---|---|---|
| `SummarizationMiddleware` | Message history compress করে context window-এর মধ্যে রাখে | Token count বা fraction threshold পার হলে |
| `HumanInTheLoopMiddleware` | Tool call execute হওয়ার আগে মানুষের approval নেয় | যে tool-এ `interrupt_on=True` সেট করা |

**সব example-এ একই `weather_db` এবং Anthropic API ব্যবহার করা হয়েছে।**

### Step 1 — Environment Setup

**কী হচ্ছে:** `.env` file থেকে `ANTHROPIC_API_KEY` load করা হচ্ছে।

**কেন:** API key কখনো code-এ লেখা যাবে না।

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


---
### Step 2 — Weather Tools এবং LLM তৈরি

**কী হচ্ছে:** দুটো tool তৈরি করা হচ্ছে:
- `get_weather` → safe read-only tool, কোনো approval লাগবে না
- `send_weather_alert` → sensitive action, Human-in-the-Loop example-এ মানুষের approval লাগবে

**কেন দুটো tool:**

| Tool | ধরন | Approval? |
|---|---|---|
| `get_weather` | Read-only | ❌ না |
| `send_weather_alert` | Side-effect (notification পাঠায়) | ✅ হ্যাঁ |

In [2]:
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic

weather_db = {
    "dhaka":      {"temperature": "34°C", "condition": "Sunny",         "humidity": "72%", "wind_speed": "10 km/h", "feels_like": "38°C"},
    "chittagong": {"temperature": "32°C", "condition": "Partly Cloudy", "humidity": "78%", "wind_speed": "14 km/h", "feels_like": "36°C"},
    "london":     {"temperature": "17°C", "condition": "Overcast",      "humidity": "85%", "wind_speed": "20 km/h", "feels_like": "15°C"},
}

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    key = city.lower().strip()
    if key not in weather_db:
        return f"No weather data available for '{city}'."
    w = weather_db[key]
    return (
        f"Weather in {city.title()}:\n"
        f"  Temperature : {w['temperature']} (feels like {w['feels_like']})\n"
        f"  Condition   : {w['condition']}\n"
        f"  Humidity    : {w['humidity']}\n"
        f"  Wind Speed  : {w['wind_speed']}"
    )

@tool
def send_weather_alert(city: str, message: str) -> str:
    """Send a weather alert notification for a city. Requires human approval."""
    return f"[ALERT SENT] {city.title()}: {message}"

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

print("Tools and LLM ready.")
print("Tools:", [get_weather.name, send_weather_alert.name])

Tools and LLM ready.
Tools: ['get_weather', 'send_weather_alert']


---
## Middleware কীভাবে কাজ করে

LangChain-এর middleware system দুটো **hook** দিয়ে কাজ করে:

| Hook | কখন চলে | ব্যবহার |
|---|---|---|
| `before_model` | প্রতিটা LLM call-এর **আগে** | History summarize, PII redact |
| `after_model` | প্রতিটা LLM call-এর **পরে** | Tool approval intercept, logging |

**একবার configure করো, সবসময় automatic:**

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model=llm,
    tools=[...],
    middleware=[
        SummarizationMiddleware(model=llm, trigger=("tokens", 4000), keep=("messages", 20)),
        HumanInTheLoopMiddleware(interrupt_on={"send_alert": {"allowed_decisions": ["approve", "reject"]}}),
    ],
)
```

**`middleware=[]` list-এ দুটো একসাথেও দেওয়া যায় — order matter করে।**

---
## 1. SummarizationMiddleware

**কী করে:** Conversation history-তে যখন অনেক বেশি message জমে token limit-এর কাছে চলে আসে, তখন পুরনো message গুলো একটা summary-তে compress করে দেয়।

**কেন দরকার:**
- LLM-এর context window সীমিত (Claude Haiku ≈ 200K tokens)
- বড় history → cost বাড়ে + slow হয়
- Summary রাখলে context টিকে থাকে কিন্তু token কম লাগে

**Trigger ধরন:**

| Trigger | মানে | উদাহরণ |
|---|---|---|
| `("tokens", N)` | N token পার হলে trigger | `("tokens", 500)` |
| `("fraction", F)` | Context window-এর F% পার হলে trigger | `("fraction", 0.8)` |
| `[(...), (...)]` | OR logic — যেকোনো একটা পার হলেই trigger | `[("tokens", 800), ("messages", 6)]` |

**Keep parameter:**

| Keep | মানে |
|---|---|
| `("messages", N)` | সর্বশেষ N টা message হুবহু রাখো |
| `("fraction", F)` | Context-এর F অংশ রাখো |
| `("tokens", N)` | N token পর্যন্ত রাখো |

### 1a. Token-based Trigger

**কী হচ্ছে:** `trigger=("tokens", 500)` — যখন conversation history **৫০০ token** ছাড়িয়ে যাবে, SummarizationMiddleware স্বয়ংক্রিয়ভাবে পুরনো message গুলো summary-তে মিশিয়ে দেবে।

**Demo-র জন্য threshold কম (৫০০ token):** কয়েকটা turn-এই trigger হবে। Production-এ সাধারণত ৪০০০–১০০০০ রাখা হয়।

**`keep=("messages", 5)` মানে:** Summarize করার পরেও সর্বশেষ ৫টা message হুবহু রাখো।

**Flow:**
```
Turn 1  → 200 tokens  (threshold-এর নিচে, কোনো summarization নেই)
Turn 2  → 420 tokens  (threshold-এর নিচে)
Turn 3  → 580 tokens  ← threshold পার! → SummarizationMiddleware fires → history compress
Turn 4  → নতুন শুরু, summary সহ
```

In [5]:
import uuid
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# ── Agent with token-based SummarizationMiddleware ──
agent_sum_token = create_agent(
    model=llm,
    tools=[get_weather],
    checkpointer=InMemorySaver(),   # get_state()-এর জন্য checkpointer আবশ্যক
    middleware=[
        SummarizationMiddleware(
            model=llm,                # summarization-এর জন্য একই model
            trigger=("tokens", 500),  # 500 token পার হলে compress
            keep=("messages", 5),     # সর্বশেষ 5টা message রাখো
        ),
    ],
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

questions = [
    "Dhaka-র আজকের আবহাওয়া কেমন? বিস্তারিত বলো।",
    "London-এর আবহাওয়া কেমন? Dhaka-র সাথে তুলনা করো।",
    "Chittagong-এর আবহাওয়া জানাও।",
    "এই তিনটা শহরের মধ্যে এখন কোথায় যাওয়া সবচেয়ে ভালো?",
]

for i, q in enumerate(questions, 1):
    print(f"\n=== Turn {i} ===")
    print(f"Human: {q}")
    result = agent_sum_token.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config,
    )
    state = agent_sum_token.get_state(config)
    msgs  = state.values.get("messages", [])
    last  = result["messages"][-1]
    preview = last.content[:120] if isinstance(last.content, str) else str(last.content)[:120]
    tokens  = (getattr(last, "usage_metadata", None) or {}).get("output_tokens", "?")
    print(f"AI: {preview}...")
    print(f"[History: {len(msgs)} messages | output_tokens: {tokens}]")

print("\n=== Final message history ===")
final_state = agent_sum_token.get_state(config)
for m in final_state.values.get("messages", []):
    role    = type(m).__name__
    preview = str(m.content)[:80].replace("\n", " ")
    print(f"  [{role}] {preview}")


=== Turn 1 ===
Human: Dhaka-র আজকের আবহাওয়া কেমন? বিস্তারিত বলো।
AI: ঢাকার আজকের আবহাওয়া সম্পর্কে বিস্তারিত তথ্য:

**🌡️ তাপমাত্রা:**
- বর্তমান তাপমাত্রা: **৩৪°C**
- অনুভূত তাপমাত্রা: **৩৮°...
[History: 4 messages | output_tokens: 480]

=== Turn 2 ===
Human: London-এর আবহাওয়া কেমন? Dhaka-র সাথে তুলনা করো।
AI: লন্ডনের আবহাওয়া এবং ঢাকার সাথে বিস্তারিত তুলনা:

## **লন্ডনের আবহাওয়া:**

**🌡️ তাপমাত্রা:**
- বর্তমান তাপমাত্রা: **১৭°...
[History: 8 messages | output_tokens: 969]

=== Turn 3 ===
Human: Chittagong-এর আবহাওয়া জানাও।
AI: ## **চট্টগ্রামের আবহাওয়া:**

**🌡️ তাপমাত্রা:**
- বর্তমান তাপমাত্রা: **৩২°C**
- অনুভূত তাপমাত্রা: **৩৬°C** (গরম অনুভব হব...
[History: 8 messages | output_tokens: 908]

=== Turn 4 ===
Human: এই তিনটা শহরের মধ্যে এখন কোথায় যাওয়া সবচেয়ে ভালো?
AI: ## **তিনটি শহরের মধ্যে কোথায় যাওয়া সবচেয়ে ভালো?**

এটি আপনার পছন্দ এবং উদ্দেশ্যের উপর নির্ভর করে। এখানে বিস্তারিত বিশ...
[History: 7 messages | output_tokens: 1374]

=== Final message history ===
  [HumanMessage]

---
### 1b. Fraction-based Trigger এবং OR Logic

**Fraction-based:** `trigger=("fraction", 0.8)` — model-এর **context window-এর ৮০%** ভরে গেলে summarize।

**Token-based থেকে পার্থক্য:**

| | Token-based | Fraction-based |
|---|---|---|
| **Trigger** | Fixed number (e.g. ৫০০) | % of context window |
| **Flexibility** | সব model-এ same value | Model বদলালে auto-adjust |
| **Use case** | Predictable | Model-agnostic deployment |

**OR Logic (list of tuples):** `trigger=[("tokens", 800), ("messages", 6)]`
- মানে: token >= 800 **বা** message count >= 6 — যেটা আগে হবে সেটাই trigger করবে
- ব্যবহার: loose threshold — দুটোর যেকোনো একটা পার হলেই summarize

```
[("tokens", 800), ("messages", 6)]
      ↑                  ↑
tokens হলে         অথবা messages হলে  →  OR
```

In [6]:
import uuid
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage

# ── Variant A: Fraction-based ──
agent_frac = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.8),   # context window-এর 80% ভরলে trigger
            keep=("fraction", 0.3),       # 30% recent context রাখো
        ),
    ],
)

# ── Variant B: OR Logic — tokens বা messages যেটা আগে পার হয় ──
agent_or = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=[("tokens", 800), ("messages", 6)],  # OR: যেকোনো একটা
            keep=("messages", 4),
        ),
    ],
)

config_frac = {"configurable": {"thread_id": str(uuid.uuid4())}}
config_or   = {"configurable": {"thread_id": str(uuid.uuid4())}}

msg = HumanMessage(content="Dhaka-র আবহাওয়া জানাও।")

res_frac = agent_frac.invoke({"messages": [msg]}, config=config_frac)
res_or   = agent_or.invoke(  {"messages": [msg]}, config=config_or)

def preview(content, n=150):
    return content[:n] if isinstance(content, str) else str(content)[:n]

print("=== Fraction-based agent ===")
print("Trigger : context window-এর 80%")
print("Keep    : context window-এর 30%")
print("Answer  :", preview(res_frac["messages"][-1].content))

print("\n=== OR-logic agent ===")
print("Trigger : tokens >= 800 OR messages >= 6")
print("Keep    : সর্বশেষ 4টা message")
print("Answer  :", preview(res_or["messages"][-1].content))

print("\n=== তিনটা variant-এর তুলনা ===")
variants = [
    ("Token-based (1a)",    "trigger=(tokens, 500)",                "keep=(messages, 5)"),
    ("Fraction-based (1b)", "trigger=(fraction, 0.8)",              "keep=(fraction, 0.3)"),
    ("OR-logic (1b)",       "trigger=[(tokens,800),(messages,6)]",  "keep=(messages, 4)"),
]
print("%-25s %-42s %s" % ("Variant", "Trigger", "Keep"))
print("-" * 82)
for name, trig, keep in variants:
    print(f"{name:<25} {trig:<42} {keep}")

=== Fraction-based agent ===
Trigger : context window-এর 80%
Keep    : context window-এর 30%
Answer  : ঢাকার বর্তমান আবহাওয়ার তথ্য:

- **তাপমাত্রা**: ৩৪°সে (অনুভূত তাপমাত্রা ৩৮°সে)
- **আবহাওয়া অবস্থা**: রৌদ্রোজ্জ্বল
- **আর্দ্রতা**: ৭২%
- **বায়ু গতি**

=== OR-logic agent ===
Trigger : tokens >= 800 OR messages >= 6
Keep    : সর্বশেষ 4টা message
Answer  : ঢাকার বর্তমান আবহাওয়া:

- **তাপমাত্রা**: ৩৪°C (অনুভূত তাপমাত্রা ৩৮°C)
- **আবহাওয়া**: রৌদ্রোজ্জ্বল
- **আর্দ্রতা**: ৭২%
- **বায়ু গতি**: ১০ কিমি/ঘন্টা

=== তিনটা variant-এর তুলনা ===
Variant                   Trigger                                    Keep
----------------------------------------------------------------------------------
Token-based (1a)          trigger=(tokens, 500)                      keep=(messages, 5)
Fraction-based (1b)       trigger=(fraction, 0.8)                    keep=(fraction, 0.3)
OR-logic (1b)             trigger=[(tokens,800),(messages,6)]        keep=(messages, 4)


---
## 2. HumanInTheLoopMiddleware

**কী করে:** Agent যখন নির্দিষ্ট tool call করতে যায়, execute হওয়ার **আগে** pause করে মানুষের approval চায়।

**কেন দরকার:**
- কিছু action irreversible — যেমন alert পাঠানো, email, database modify করা
- AI ভুল করলে ধরার সুযোগ দেয়
- Compliance ও audit-এর জন্য human review mandatory হতে পারে

**Flow:**
```
Agent: "send_weather_alert call করবো"
             ↓
HumanInTheLoopMiddleware intercepts (after_model hook)
             ↓
Execution PAUSED — agent.invoke() return করে
result-এ "__interrupt__" key থাকে
             ↓
Human দেখে: tool name, args → সিদ্ধান্ত নেয়
             ↓
Command(resume={"decisions": [{"type": "approve"}]})
             ↓
Tool execute হয়, agent চালিয়ে যায়
```

**Decision types:**

| Decision | মানে |
|---|---|
| `"approve"` | Tool হুবহু execute করো |
| `"reject"` | Execute করো না, agent-কে জানাও |
| `"edit"` | Args পরিবর্তন করে তারপর execute করো |
| `"respond"` | Human নিজেই tool-এর মতো একটা response দেয় |

**Required:** `checkpointer=InMemorySaver()` — interrupt state সংরক্ষণ করে, তাই resume করা যায়।

In [7]:
import uuid
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# ── Checkpointer: interrupt state persist করতে (HITL-এর জন্য আবশ্যক) ──
checkpointer = InMemorySaver()

# ── Agent with HumanInTheLoopMiddleware ──
agent_hitl = create_agent(
    model=llm,
    tools=[get_weather, send_weather_alert],
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # send_weather_alert: approve বা reject করা যাবে
                "send_weather_alert": {
                    "allowed_decisions": ["approve", "reject"],
                },
                # get_weather: safe tool, interrupt নেই → auto-approve
                "get_weather": False,
            }
        ),
    ],
)

print("Agent with HumanInTheLoopMiddleware ready.")
print("Interrupt on  : send_weather_alert")
print("  Decisions   : approve | reject")
print("No interrupt  : get_weather (auto-approved)")

Agent with HumanInTheLoopMiddleware ready.
Interrupt on  : send_weather_alert
  Decisions   : approve | reject
No interrupt  : get_weather (auto-approved)


In [8]:
# ── Full HITL Flow: Approve ──

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print("=" * 60)
print("Turn 1: Human asks agent to check weather and send alert")
print("=" * 60)

user_msg = (
    "Dhaka-র আবহাওয়া চেক করো। "
    "যদি তাপমাত্রা 30°C-এর বেশি হয় তাহলে একটা heat alert পাঠাও।"
)
print(f"Human: {user_msg}\n")

result1 = agent_hitl.invoke(
    {"messages": [HumanMessage(content=user_msg)]},
    config=config,
)

# ── Interrupt detection ──
# invoke() return করে — result-এ "__interrupt__" key থাকে যদি interrupt হয়
interrupt_data = result1.get("__interrupt__")

if interrupt_data:
    hitl_value = interrupt_data[0].value    # HITLRequest
    action     = hitl_value["action_requests"][0]

    print(">>> EXECUTION PAUSED — Human approval required!")
    tool_name = action["name"]
    tool_args = action["args"]
    tool_desc = action["description"]
    print(">>> EXECUTION PAUSED — Human approval required!")
    print(f"    Tool name : {tool_name}")
    print(f"    Args      : {tool_args}")
    print(f"    Desc      : {tool_desc[:80]}...")

    # ── Human reviews ──
    print("\n" + "=" * 60)
    print("Human Review")
    print("=" * 60)
    human_decision = "approve"   # <-- মানুষ এখানে সিদ্ধান্ত নেয়
    print(f"Human decision: \"{human_decision}\"\n")

    # ── Resume ──
    # decisions = list — প্রতিটা intercepted tool call-এর জন্য একটা entry
    print("=" * 60)
    print("Turn 2: Resuming with human decision")
    print("=" * 60)

    result2 = agent_hitl.invoke(
        Command(resume={"decisions": [{"type": human_decision}]}),
        config=config,
    )

    content = result2["messages"][-1].content
    print("Agent final answer:")
    print(content if isinstance(content, str) else str(content))

else:
    print("No interrupt — execution completed without human approval.")
    content = result1["messages"][-1].content
    print("Final:", content if isinstance(content, str) else str(content))

Turn 1: Human asks agent to check weather and send alert
Human: Dhaka-র আবহাওয়া চেক করো। যদি তাপমাত্রা 30°C-এর বেশি হয় তাহলে একটা heat alert পাঠাও।

>>> EXECUTION PAUSED — Human approval required!
>>> EXECUTION PAUSED — Human approval required!
    Tool name : send_weather_alert
    Args      : {'city': 'Dhaka', 'message': 'Heat Alert: Dhaka-তে অত্যন্ত গরম। বর্তমান তাপমাত্রা 34°C (যা 38°C-এর মতো অনুভূত হচ্ছে)। প্রচুর পানি পান করুন এবং প্রয়োজনে ঘরের ভিতরে থাকুন।'}
    Desc      : Tool execution requires approval

Tool: send_weather_alert
Args: {'city': 'Dhaka...

Human Review
Human decision: "approve"

Turn 2: Resuming with human decision
Agent final answer:
✅ **সম্পন্ন!** 

**Dhaka-র আবহাওয়া:**
- 🌡️ **তাপমাত্রা**: 34°C (অনুভূত: 38°C)
- ☀️ **অবস্থা**: রৌদ্রোজ্জ্বল
- 💧 **আর্দ্রতা**: 72%
- 💨 **বাতাসের গতি**: 10 km/h

যেহেতু তাপমাত্রা 30°C-এর বেশি ছিল, আমি একটি **Heat Alert** সফলভাবে পাঠিয়েছি। সবাইকে সাবধান থাকতে বলা হয়েছে যাতে তারা প্রচুর পানি পান করে এবং গরম থেকে সুরক্ষিত থাকে।


In [9]:
# ── HITL: Reject Flow ──
# Human যদি reject করে তাহলে agent কী করে?

thread_id_reject = str(uuid.uuid4())
config_reject = {"configurable": {"thread_id": thread_id_reject}}

print("=== Reject Flow ===")
print("Human: Dhaka-র জন্য weather alert পাঠাও।\n")

result_r = agent_hitl.invoke(
    {"messages": [HumanMessage(content="Dhaka-র জন্য weather alert পাঠাও।")]},
    config=config_reject,
)

interrupt_data_r = result_r.get("__interrupt__")

if interrupt_data_r:
    action_r = interrupt_data_r[0].value["action_requests"][0]
    print(">>> PAUSED — Human review needed")
    r_name = action_r["name"]
    r_args = action_r["args"]
    print(">>> PAUSED — Human review needed")
    print(f"    Tool : {r_name}")
    print(f"    Args : {r_args}")

    print("\nHuman decides: \"reject\"\n")

    result_rejected = agent_hitl.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config_reject,
    )
    content = result_rejected["messages"][-1].content
    print("Agent response after rejection:")
    print(content if isinstance(content, str) else str(content))

else:
    print("No interrupt raised.")

print("\n=== Decision type তুলনা ===")
decisions = [
    ("approve", "Tool-কে হুবহু execute করো"),
    ("reject",  "Tool execute করো না; agent-কে জানাও"),
    ("edit",    "Args বদলে তারপর execute করো"),
    ("respond", "Human নিজেই tool-এর উত্তর দেয়"),
]
print("%-12s %s" % ("Decision", "মানে"))
print("-" * 50)
for d, meaning in decisions:
    print(f"{d:<12} {meaning}")

=== Reject Flow ===
Human: Dhaka-র জন্য weather alert পাঠাও।

No interrupt raised.

=== Decision type তুলনা ===
Decision     মানে
--------------------------------------------------
approve      Tool-কে হুবহু execute করো
reject       Tool execute করো না; agent-কে জানাও
edit         Args বদলে তারপর execute করো
respond      Human নিজেই tool-এর উত্তর দেয়


---
## সারসংক্ষেপ

| Middleware | Import | দরকারি parameter | Checkpointer? |
|---|---|---|---|
| `SummarizationMiddleware` | `langchain.agents.middleware` | `model`, `trigger`, `keep` | ❌ optional |
| `HumanInTheLoopMiddleware` | `langchain.agents.middleware` | `interrupt_on` | ✅ required |

**Trigger তুলনা (Summarization):**

| Trigger type | Example | Behavior |
|---|---|---|
| Token count | `("tokens", 500)` | 500 token হলেই summarize |
| Fraction | `("fraction", 0.8)` | Context-এর 80% ভরলে |
| OR logic | `[("tokens", 800), ("messages", 6)]` | যেকোনো একটা পার হলেই |

**HITL Full Flow:**
```python
# Step 1: invoke → interrupt হলে result-এ "__interrupt__" key থাকে
result = agent.invoke({"messages": [...]}, config)

# Step 2: interrupt detect করো
interrupt_data = result.get("__interrupt__")
if interrupt_data:
    action = interrupt_data[0].value["action_requests"][0]
    print(action["name"], action["args"])

# Step 3: human সিদ্ধান্ত নেয় → resume
result2 = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),  # বা "reject"
    config
)
```

**কোনটা কখন:**
- Long conversation, cost বাঁচাতে → **SummarizationMiddleware**
- Sensitive/irreversible action, compliance → **HumanInTheLoopMiddleware**
- দুটো একসাথে → `middleware=[SummarizationMiddleware(...), HumanInTheLoopMiddleware(...)]`